# Phase 6: Final Evaluation, SHAP Analysis, and Business Impact

**Predicting 30-Day Hospital Readmission for Diabetic Patients**

Evaluate the tuned XGBoost model on the held-out test set. Test set is unlocked here for the first and only time. Includes: full metric suite, ROC/PR curves, SHAP analysis, threshold optimization, cost simulation.

**Inputs:** Phase 3 data + Phase 5 tuned models
**Outputs:** Final test metrics, SHAP values, business impact simulation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, pickle, warnings, time, os
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

COLORS = {
    'primary': '#1B2A4A', 'steel': '#4A7FB5', 'teal': '#2AA198',
    'orange': '#E67E22', 'red': '#E74C3C', 'green': '#2ECC71', 'gold': '#D4A017'
}

ARTIFACTS_DIR = '../artifacts'
os.makedirs('../figures', exist_ok=True)
os.makedirs('../models', exist_ok=True)
print('Environment ready.')

### Load Phase 3 + Phase 5 Artifacts

**Test set unlocks here -- run once, do not tune after this point.**

In [ ]:
# Load Phase 3 data
X_train = pd.read_parquet(f'{ARTIFACTS_DIR}/phase3_X_train.parquet')
X_test  = pd.read_parquet(f'{ARTIFACTS_DIR}/phase3_X_test.parquet')
y_train = pd.read_parquet(f'{ARTIFACTS_DIR}/phase3_y_train.parquet').squeeze()
y_test  = pd.read_parquet(f'{ARTIFACTS_DIR}/phase3_y_test.parquet').squeeze()
preprocessor = joblib.load(f'{ARTIFACTS_DIR}/phase3_preprocessor.joblib')

with open(f'{ARTIFACTS_DIR}/phase3_meta.pkl', 'rb') as f:
    p3_meta = pickle.load(f)
all_features = p3_meta['all_features']

# Load Phase 5 best model and all tuned models
final_model = joblib.load(f'{ARTIFACTS_DIR}/phase5_best_model.joblib')

with open(f'{ARTIFACTS_DIR}/phase5_meta.pkl', 'rb') as f:
    p5_meta = pickle.load(f)

tuned_models = {}
for name in p5_meta.get('tuned_model_names', []):
    safe = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    fpath = f'{ARTIFACTS_DIR}/phase5_tuned_{safe}.joblib'
    if os.path.exists(fpath):
        tuned_models[name] = joblib.load(fpath)

print(f'Artifacts loaded')
print(f'  Best model: {p5_meta["best_model_name"]}')
print(f'  CV ROC-AUC: {p5_meta["best_cv_roc_auc"]:.4f}')
print(f'  Tuned models: {list(tuned_models.keys())}')
print(f'\n  TEST SET IS NOW UNLOCKED -- final evaluation only')

### Utility Functions

Redefine the utility functions from Phase 4 (they don't persist across notebooks).

In [ ]:
# === Utility Functions for Modeling Phases ===
# Defined once here, reused across Phases 4, 5, and 6.

from sklearn.metrics import (roc_auc_score, recall_score, precision_score,
                             f1_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, precision_recall_curve)


def evaluate_model(y_true, y_pred, y_prob, model_name="Model", show_matrix=True):
    """Compute and print standard classification metrics.

    Args:
        y_true: actual labels
        y_pred: predicted labels (binary)
        y_prob: predicted probabilities for positive class
        model_name: display name for printing
        show_matrix: whether to plot confusion matrix

    Returns:
        dict of metric values
    """
    metrics = {
        'auc': roc_auc_score(y_true, y_prob),
        'recall': recall_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred),
    }

    print(f"\n{'=' * 50}")
    print(f"{model_name} -- Evaluation Metrics")
    print(f"{'=' * 50}")
    print(f"  ROC-AUC:   {metrics['auc']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  F1-Score:  {metrics['f1']:.4f}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['Not Readmit', 'Readmit <30'])}")

    if show_matrix:
        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay.from_predictions(
            y_true, y_pred,
            display_labels=['Not Readmit', 'Readmit <30'],
            cmap='Blues', ax=ax, values_format=','
        )
        ax.set_title(f'{model_name} -- Confusion Matrix', fontweight='bold')
        plt.tight_layout()
        plt.show()

    return metrics


def predict_at_threshold(y_prob, threshold=0.5):
    """Convert probabilities to binary predictions at a custom threshold.

    Args:
        y_prob: predicted probabilities for positive class
        threshold: classification cutoff (default 0.5)

    Returns:
        numpy array of binary predictions
    """
    return (y_prob >= threshold).astype(int)


def compute_business_impact(y_true, y_prob, threshold,
                            penalty_saved_per_tp=13000,
                            intervention_cost_per_flag=500):
    """Compute cost-adjusted net benefit at a given threshold.

    Simulates the financial impact of deploying the model at a hospital
    with the given cost assumptions.

    Args:
        y_true: actual labels
        y_prob: predicted probabilities
        threshold: classification cutoff
        penalty_saved_per_tp: estimated $ saved per correctly flagged readmission
        intervention_cost_per_flag: $ cost per patient flagged (TP + FP)

    Returns:
        dict with TP, FP, FN, net_benefit, and per-patient metrics
    """
    y_pred = predict_at_threshold(y_prob, threshold)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    total_flagged = tp + fp
    savings = tp * penalty_saved_per_tp
    costs = total_flagged * intervention_cost_per_flag
    net_benefit = savings - costs

    return {
        'threshold': threshold,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'total_flagged': total_flagged,
        'recall': tp / max(tp + fn, 1),
        'precision': tp / max(tp + fp, 1),
        'savings': savings,
        'costs': costs,
        'net_benefit': net_benefit,
    }


print("Utility functions defined:")
print("  evaluate_model()          -- metrics + classification report + optional confusion matrix")
print("  predict_at_threshold()    -- probability to binary at custom cutoff")
print("  compute_business_impact() -- cost-adjusted net benefit simulation")

### 6.1 Unlock Test Set and Generate Predictions

This is the first and only time the test set is used. The final XGBoost model was selected and fitted on the full training set in Phase 5.

In [ ]:
import joblib

# Load final model (or use in-memory if available)
final_model = joblib.load('../models/final_model.joblib')

# Generate predictions at default 0.5 threshold
y_prob = final_model.predict_proba(X_test)[:, 1]
y_pred = predict_at_threshold(y_prob, threshold=0.5)

print("=" * 60)
print("TEST SET EVALUATION -- FIRST AND ONLY USE")
print("=" * 60)

test_metrics = evaluate_model(y_test, y_pred, y_prob,
                              model_name="XGBoost (tuned) -- Test Set",
                              show_matrix=True)

The tuned XGBoost model achieved a test-set AUC of 0.6930, compared to the CV AUC of 0.7025 -- a drop of less than 0.01, indicating strong generalization with no meaningful overfitting. Both the AUC > 0.65 target and the Recall > 0.50 target are met on unseen data (Recall = 0.6211). At the default 0.5 threshold, the model correctly identified 613 of 987 readmitted patients (true positives) while generating 4,577 false positives out of 13,101 non-readmitted patients. Precision remains low at 11.8%, which is expected given the ~7% prevalence rate and the use of class_weight balancing -- the model intentionally trades precision for recall to catch more readmissions. The 374 missed readmissions (false negatives) represent patients who would not receive a preventive intervention under the current threshold; threshold optimization in Section 6.4 explores how to reduce this number.

### 6.2 ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

colors_list = [COLORS['steel'], COLORS['green'], COLORS['orange']]

for i, (name, pipeline) in enumerate(tuned_models.items()):
    prob = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    lw = 3 if 'XGB' in name else 1.5
    ax.plot(fpr, tpr, linewidth=lw, color=colors_list[i],
            label=f'{name} (AUC={auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.5000)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curves -- Test Set (All Tuned Models)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig('../figures/roc_curves_test.png', dpi=150, bbox_inches='tight')
plt.show()

All three tuned models performed similarly on the test set: XGBoost led with AUC 0.6930, followed closely by Random Forest (0.6908) and Logistic Regression (0.6905). The tight clustering -- all within 0.003 of each other -- suggests that linear and nonlinear models extract comparable signal from this feature set. All three curves sit well above the random baseline diagonal, confirming the models learned real predictive signal. XGBoost's thicker curve shows a slight advantage in the mid-range of the ROC space, consistent with its ability to capture the nonlinear interactions identified during EDA.

### 6.3 Precision-Recall Curve

In [ ]:
from sklearn.metrics import average_precision_score

precision_vals, recall_vals, pr_thresholds = precision_recall_curve(y_test, y_prob)
avg_precision = average_precision_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(recall_vals, precision_vals, color=COLORS['orange'], linewidth=2,
        label=f'XGBoost (AP={avg_precision:.4f})')
ax.axhline(y=y_test.mean(), color=COLORS['red'], linestyle='--',
           label=f'Baseline (prevalence={y_test.mean():.3f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve -- Test Set', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig('../figures/precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.show()

The precision-recall curve provides a more honest view of model performance than ROC for imbalanced data. With only 7.0% prevalence, the ROC curve looks optimistic because the large negative class keeps the false positive rate low even when thousands of false positives occur. The PR curve directly shows the tradeoff the hospital faces: at high recall (catching most readmissions), precision drops to near baseline. The average precision (AP) of 0.1565 is roughly 2.2x the baseline prevalence of 0.070, confirming the model adds meaningful signal above random. The steep drop-off in precision beyond ~20% recall illustrates why the default 0.5 threshold yields only 11.8% precision -- the model must flag many patients to catch a majority of true readmissions.

### 6.4 Threshold Optimization

The default 0.5 threshold is not appropriate for imbalanced classes. We sweep thresholds to find the operating point that maximizes F1 and identify the threshold needed to achieve at least 70% recall.

In [ ]:
thresholds = np.arange(0.05, 0.51, 0.01)
threshold_results = []

for t in thresholds:
    preds = predict_at_threshold(y_prob, t)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    rec = tp / max(tp + fn, 1)
    prec = tp / max(tp + fp, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-8)
    threshold_results.append({
        'threshold': t, 'recall': rec, 'precision': prec,
        'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'flagged': tp + fp
    })

threshold_df = pd.DataFrame(threshold_results)

# Find optimal thresholds
best_f1_idx = threshold_df['f1'].idxmax()
best_f1_threshold = threshold_df.loc[best_f1_idx, 'threshold']
best_f1_value = threshold_df.loc[best_f1_idx, 'f1']

# Find threshold for recall >= 0.70
recall_70 = threshold_df[threshold_df['recall'] >= 0.70]
recall_70_threshold = recall_70['threshold'].max() if len(recall_70) > 0 else None

print(f"Optimal F1 threshold: {best_f1_threshold:.2f} (F1={best_f1_value:.4f})")
if recall_70_threshold:
    row = threshold_df[threshold_df['threshold'] == recall_70_threshold].iloc[0]
    print(f"Recall >= 0.70 threshold: {recall_70_threshold:.2f} (Recall={row['recall']:.3f}, Precision={row['precision']:.3f})")
else:
    print("No threshold achieved recall >= 0.70 in the 0.05-0.50 range.")

# Print key thresholds table
key_thresholds = [0.05, 0.10, 0.15, 0.20, best_f1_threshold, 0.30, 0.40, 0.50]
# deduplicate and sort
key_thresholds = sorted(set([round(t, 2) for t in key_thresholds]))

print(f"\n{'Threshold':>10} {'Recall':>8} {'Precision':>10} {'F1':>8} {'Flagged':>10} {'TP':>6} {'FP':>6} {'FN':>6}")
print("-" * 70)
for t in key_thresholds:
    row = threshold_df.iloc[(threshold_df['threshold'] - t).abs().idxmin()]
    marker = " <-- best F1" if abs(row['threshold'] - best_f1_threshold) < 0.005 else ""
    print(f"{row['threshold']:>10.2f} {row['recall']:>8.3f} {row['precision']:>10.3f} {row['f1']:>8.3f} "
          f"{row['flagged']:>10,.0f} {row['tp']:>6,.0f} {row['fp']:>6,.0f} {row['fn']:>6,.0f}{marker}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Recall, Precision, F1 vs threshold
ax1.plot(threshold_df['threshold'], threshold_df['recall'],
         color=COLORS['teal'], linewidth=2, label='Recall')
ax1.plot(threshold_df['threshold'], threshold_df['precision'],
         color=COLORS['orange'], linewidth=2, label='Precision')
ax1.plot(threshold_df['threshold'], threshold_df['f1'],
         color=COLORS['primary'], linewidth=2, label='F1')
ax1.axvline(x=best_f1_threshold, color=COLORS['red'], linestyle='--',
            linewidth=1.5, label=f'Best F1 ({best_f1_threshold:.2f})')
ax1.set_xlabel('Threshold', fontsize=12)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Metrics vs Classification Threshold', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.set_xlim([0.05, 0.50])
ax1.set_ylim([0, 1])

# Right panel: Number of patients flagged vs threshold
ax2.bar(threshold_df['threshold'], threshold_df['flagged'],
        width=0.008, color=COLORS['steel'], alpha=0.8)
ax2.axvline(x=best_f1_threshold, color=COLORS['red'], linestyle='--',
            linewidth=1.5, label=f'Best F1 ({best_f1_threshold:.2f})')
ax2.set_xlabel('Threshold', fontsize=12)
ax2.set_ylabel('Patients Flagged', fontsize=12)
ax2.set_title('Patients Flagged vs Threshold', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.set_xlim([0.05, 0.50])

plt.tight_layout()
plt.savefig('../figures/threshold_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

The optimal F1 threshold is 0.50 (F1 = 0.1985), which coincidentally matches the default. This is unusual for imbalanced data and reflects the fact that XGBoost's scale_pos_weight already shifts the model's internal calibration toward the minority class. Lowering the threshold to 0.46 achieves recall >= 0.70 (catching 711 of 987 readmissions) but increases the number of patients flagged substantially. The left panel shows that recall remains above 0.90 down to threshold 0.30, while precision stays below 0.10 across most of the range -- a reflection of the ~7% base rate. The right panel shows the exponential increase in flagged patients as the threshold drops: from ~5,200 at 0.50 to over 14,000 at 0.05. The hospital's choice of threshold depends on their cost structure -- whether missing a readmission ($13,000 penalty) outweighs the cost of unnecessary interventions ($500 each). This is quantified in Section 6.7.

### 6.5 SHAP Analysis

SHAP (SHapley Additive exPlanations) uses game theory to assign each feature a contribution to each prediction. We use TreeExplainer, which is exact and fast for tree-based models like XGBoost.

In [ ]:
import shap

# Extract the XGBoost classifier from the pipeline
xgb_classifier = final_model.named_steps['classifier']

# Transform test features through the preprocessor
X_test_transformed = final_model.named_steps['preprocessor'].transform(X_test)
feature_names_out = final_model.named_steps['preprocessor'].get_feature_names_out()

# Use a sample if full test set is too large for SHAP
n_shap = min(3000, X_test_transformed.shape[0])
shap_sample_idx = np.random.RandomState(42).choice(
    X_test_transformed.shape[0], n_shap, replace=False)
X_shap = X_test_transformed[shap_sample_idx]

# Convert sparse matrix to dense if needed
if hasattr(X_shap, 'toarray'):
    X_shap = X_shap.toarray()

# Compute SHAP values
explainer = shap.TreeExplainer(xgb_classifier)
shap_values = explainer.shap_values(X_shap)

print(f"SHAP values computed for {n_shap} test samples")
print(f"Shape: {shap_values.shape}")
print(f"Features: {len(feature_names_out)}")

In [ ]:
# SHAP Visualization 1: Summary Bar Plot (Global Feature Importance)
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=feature_names_out,
                  plot_type="bar", max_display=20, show=False)
plt.title('Top 20 Features by Mean |SHAP Value|', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

The SHAP bar plot reveals that **time_in_hospital** is the single most important feature (mean |SHAP| ~0.28), followed closely by **number_inpatient** (~0.22). This partially confirms the EDA hypothesis that prior utilization would dominate -- number_inpatient ranks #2, and the engineered **total_visits_prior_year** appears at #7. Discharge disposition categories collectively rank highly (#3, #6, #9), confirming their importance. Notably, **num_lab_procedures** (#4) and **number_diagnoses** (#5) -- proxies for clinical complexity -- also feature prominently. The engineered feature **diabetesMed_Yes** ranks #8 in the top 10. However, **has_A1c_measured** does not appear in the top 20, which contradicts the EDA hypothesis -- its signal may have been absorbed by correlated features in the tree ensemble.

In [ ]:
# SHAP Visualization 2: Summary Dot Plot (Directional Effects)
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=feature_names_out,
                  max_display=20, show=False)
plt.title('SHAP Value Distribution by Feature', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/shap_dot_plot.png', dpi=150, bbox_inches='tight')
plt.show()

The dot plot reveals clear directional patterns for the top features. **number_inpatient** shows a strong rightward spread for high values (red dots), meaning patients with more prior inpatient visits are pushed toward higher readmission probability -- consistent with the EDA finding that prior utilization is the strongest clinical signal. **time_in_hospital** shows a complex pattern: shorter stays (blue/low) push predictions in both directions, while longer stays (red/high) tend to push predictions leftward (lower risk), possibly reflecting that longer stays allow more complete treatment. **discharge_disposition_id_1** (discharged to home) shows red dots pulling left -- being sent home is associated with lower readmission risk compared to skilled nursing or rehab facilities. **discharge_disposition_id_3** (skilled nursing facility) shows the opposite pattern, with high values pushing toward readmission. **total_visits_prior_year** follows number_inpatient's pattern: higher prior utilization (red) pushes predictions rightward toward readmission.

In [ ]:
# SHAP Visualization 3: Waterfall Plots for Individual Patients
# Select 3 patients from the SHAP sample: high-risk, low-risk, borderline

# Get probabilities for the SHAP sample patients
shap_probs = y_prob[shap_sample_idx]

high_risk_idx = np.argmax(shap_probs)
low_risk_idx = np.argmin(shap_probs)
borderline_idx = np.argmin(np.abs(shap_probs - best_f1_threshold))

for label, idx in [('High-Risk Patient', high_risk_idx),
                   ('Low-Risk Patient', low_risk_idx),
                   ('Borderline Patient', borderline_idx)]:
    print(f"\n{'=' * 50}")
    print(f"{label}")
    print(f"Predicted probability: {shap_probs[idx]:.3f}")
    print(f"{'=' * 50}")

    explanation = shap.Explanation(
        values=shap_values[idx],
        base_values=explainer.expected_value,
        data=X_shap[idx],
        feature_names=list(feature_names_out))

    # Save the high-risk waterfall
    if label == 'High-Risk Patient':
        shap.waterfall_plot(explanation, max_display=12, show=False)
        plt.tight_layout()
        plt.savefig('../figures/shap_waterfall_high_risk.png', dpi=150, bbox_inches='tight')
        plt.show()
    else:
        shap.waterfall_plot(explanation, max_display=12, show=True)

The waterfall plots illustrate patient-level model explanations -- exactly what a clinician would see at discharge time. The **high-risk patient** (predicted probability 0.896) was driven primarily by number_inpatient = 6.748 (contributing +1.48 to the log-odds), indicating extensive prior hospitalization history, along with admission_type_id_6 (+0.4) and total_visits_prior_year (+0.32). Time_in_hospital provided a protective effect (-1.18), partially offsetting the risk factors. The **low-risk patient** (probability 0.098) lacked the high-risk utilization history, with features collectively pushing predictions well below the base rate. The **borderline patient** (probability 0.500, right at the decision threshold) shows a balanced tug-of-war between risk-increasing features (e.g., prior visits) and protective features (e.g., favorable discharge disposition), demonstrating how the model resolves ambiguous cases.

### 6.6 Clinical Risk Factor Summary

Based on the SHAP analysis, the top risk factors for 30-day readmission in diabetic inpatients are:

1. **Length of stay (time_in_hospital)** -- The single most influential feature (mean |SHAP| ~0.28). Shorter stays are associated with higher readmission risk, possibly reflecting premature discharge or less complete treatment. Longer stays tend to be protective, suggesting adequate inpatient management reduces bounce-back risk.

2. **Prior inpatient visits (number_inpatient)** -- The second strongest predictor (mean |SHAP| ~0.22). Each additional prior inpatient visit in the preceding year substantially increases predicted readmission probability. Patients with 2+ prior admissions are at markedly elevated risk. This is the most actionable utilization-based signal.

3. **Discharge disposition** -- Where patients go after discharge matters considerably. Discharge to home (disposition_id_1) is associated with lower readmission risk, while discharge to skilled nursing facilities (disposition_id_3) or other inpatient facilities (disposition_id_22) is associated with higher risk. This likely reflects underlying patient acuity rather than a causal effect of the discharge setting.

4. **Number of lab procedures (num_lab_procedures)** -- Higher lab utilization during the stay is associated with increased readmission risk, likely serving as a proxy for clinical complexity and diagnostic uncertainty rather than a direct cause.

5. **Number of diagnoses (number_diagnoses)** -- More comorbidities recorded during the encounter are associated with higher readmission probability, consistent with the clinical understanding that multimorbid patients face greater post-discharge complications.

6. **Total prior-year visits (total_visits_prior_year)** -- This engineered composite of inpatient, outpatient, and emergency visits reinforces the prior utilization signal. Frequent healthcare utilizers are at elevated readmission risk.

7. **Diabetes medication prescribed (diabetesMed_Yes)** -- Patients actively prescribed diabetes medications show a different risk profile, possibly reflecting more advanced disease requiring pharmacological management.

8. **Primary diagnosis category** -- Circulatory and respiratory primary diagnoses (diag_1_Circulatory, diag_1_Respiratory) are associated with modestly elevated readmission risk compared to other diagnosis groups.

9. **Admission type** -- Emergency/urgent admissions (admission_type_id_6) are associated with higher readmission risk compared to elective admissions, likely reflecting acuity and lack of pre-admission optimization.

10. **Age group** -- Patients aged 50-60 and 70-80 show modestly elevated readmission risk, though the effect is smaller than utilization-based features.

**Actionable implications for hospital operations:**
- **Target high-utilizers for transitional care:** Patients with 2+ inpatient visits in the prior year should automatically trigger a discharge planning consultation and post-discharge follow-up call within 48 hours.
- **Scrutinize short stays:** Very short hospital stays for diabetic patients may warrant a discharge readiness checklist to ensure treatment completion before release.
- **Coordinate post-discharge care by disposition type:** Patients discharged to SNFs or other facilities need warm handoffs with medication reconciliation, as their discharge disposition is a strong risk signal.
- **Flag complex patients:** Those with high lab utilization or 5+ diagnoses during the encounter should receive enhanced discharge education and follow-up scheduling.

**Important caveat:** These associations are observational, not causal. The dataset lacks social determinants of health (housing stability, transportation, caregiver support) that are known strong predictors of readmission. Model predictions should supplement, not replace, clinical judgment.

### 6.7 Business Impact Simulation

We simulate the financial impact of deploying the model at a hospital with 10,000 annual Medicare discharges. Cost assumptions: $500 per flagged patient for transitional care intervention, $13,000 penalty saved per correctly identified readmission (average Medicare payment x average HRRP penalty rate).

In [ ]:
# Business impact at multiple thresholds
# Scale factor: simulate 10,000 discharges (test set is ~14K, scale proportionally)
scale_factor = 10000 / len(y_test)

impact_thresholds = [0.05, 0.10, 0.15, 0.20, best_f1_threshold, 0.30, 0.40, 0.50]
impact_thresholds = sorted(set([round(t, 2) for t in impact_thresholds]))

impact_results = []
for t in impact_thresholds:
    impact = compute_business_impact(y_test, y_prob, threshold=t)
    # Scale to 10,000 discharges
    impact_results.append({
        'Threshold': t,
        'Patients Flagged': int(impact['total_flagged'] * scale_factor),
        'True Positives': int(impact['tp'] * scale_factor),
        'False Positives': int(impact['fp'] * scale_factor),
        'Missed Readmissions': int(impact['fn'] * scale_factor),
        'Recall': impact['recall'],
        'Precision': impact['precision'],
        'Intervention Cost': int(impact['total_flagged'] * scale_factor * 500),
        'Penalty Savings': int(impact['tp'] * scale_factor * 13000),
        'Net Benefit': int(impact['net_benefit'] * scale_factor),
    })

impact_df = pd.DataFrame(impact_results)
print("BUSINESS IMPACT SIMULATION")
print("Hospital: 10,000 annual Medicare discharges")
print("Intervention cost: $500/flagged patient")
print("Penalty saved: $13,000/correctly identified readmission")
print("=" * 100)

# Print with dollar formatting
for _, row in impact_df.iterrows():
    marker = " <-- best F1" if abs(row['Threshold'] - best_f1_threshold) < 0.005 else ""
    print(f"  Threshold={row['Threshold']:.2f}  |  Flagged={row['Patients Flagged']:>5,}  "
          f"TP={row['True Positives']:>4,}  FP={row['False Positives']:>5,}  "
          f"Missed={row['Missed Readmissions']:>4,}  |  "
          f"Cost=${row['Intervention Cost']:>10,}  "
          f"Savings=${row['Penalty Savings']:>10,}  "
          f"Net=${row['Net Benefit']:>10,}{marker}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Net benefit vs threshold (green if positive, red if negative)
net_benefits = impact_df['Net Benefit'].values
bar_colors = [COLORS['green'] if nb >= 0 else COLORS['red'] for nb in net_benefits]
ax1.bar(impact_df['Threshold'], net_benefits, width=0.03, color=bar_colors, alpha=0.85)
ax1.axhline(y=0, color='black', linewidth=0.8, linestyle='-')
ax1.axvline(x=best_f1_threshold, color=COLORS['red'], linestyle='--',
            linewidth=1.5, label=f'Best F1 ({best_f1_threshold:.2f})')
ax1.set_xlabel('Threshold', fontsize=12)
ax1.set_ylabel('Net Benefit ($)', fontsize=12)
ax1.set_title('Net Benefit vs Threshold', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
# Format y-axis with dollar signs
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# Right panel: Stacked comparison -- intervention cost vs penalty savings
x_pos = np.arange(len(impact_df))
width = 0.35
ax2.bar(x_pos - width/2, impact_df['Penalty Savings'], width,
        color=COLORS['green'], alpha=0.85, label='Penalty Savings')
ax2.bar(x_pos + width/2, impact_df['Intervention Cost'], width,
        color=COLORS['orange'], alpha=0.85, label='Intervention Cost')
ax2.set_xlabel('Threshold', fontsize=12)
ax2.set_ylabel('Amount ($)', fontsize=12)
ax2.set_title('Savings vs Cost by Threshold', fontsize=13, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'{t:.2f}' for t in impact_df['Threshold']], fontsize=9)
ax2.legend(fontsize=10)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('../figures/business_impact.png', dpi=150, bbox_inches='tight')
plt.show()

The business impact simulation shows that the model generates positive net benefit across all thresholds tested, with the optimal operating point at **threshold 0.30**, yielding approximately **$4.6 million in annual net benefit** for a 10,000-discharge hospital. At this threshold, the model catches 643 true readmissions (recall ~0.92) while flagging 7,498 total patients, at an intervention cost of $3.75M against $8.37M in penalty savings. The net benefit remains robust across a wide range of thresholds -- even at the most aggressive setting (0.05), net benefit is $4.1M, and at the conservative default (0.50), it is still $3.8M. No break-even threshold exists within the tested range; the 26:1 ratio of penalty saved ($13,000) to intervention cost ($500) means that even a modest true-positive rate justifies intervention. **Recommendation:** A hospital deploying this model should operate at threshold 0.30, where it captures over 90% of readmissions with a net benefit of approximately $4.6 million annually. The exact threshold can be adjusted based on the hospital's intervention capacity and staffing constraints.

### 6.8 Revisiting EDA Hypotheses

In Section 2.10, four modeling hypotheses were stated based on EDA findings. Here is how each fared:

| Hypothesis | Confirmed? | Evidence |
|------------|-----------|----------|
| Prior utilization features will dominate feature importance | **Partially** | number_inpatient ranked #2 and total_visits_prior_year ranked #7 by SHAP importance, but time_in_hospital (a stay characteristic, not prior utilization) ranked #1. Utilization features are prominent but do not solely dominate. |
| has_A1c_measured will surface in top 10 | **No** | has_A1c_measured did not appear in the top 20 SHAP features. Its signal may have been absorbed by correlated features (diabetesMed, number_diagnoses) in the tree ensemble, or the univariate EDA effect did not survive multivariate modeling. |
| Discharge disposition will matter more than admission type | **Yes** | Three discharge_disposition categories appeared in the top 10 SHAP features (#3, #6, #9), while admission_type_id_6 ranked #12. Discharge destination is a stronger predictor of readmission than how the patient was admitted. |
| Tree-based models will outperform LR | **Yes (marginal)** | XGBoost test AUC 0.6930 vs LR test AUC 0.6905 -- a consistent but small advantage of 0.0025. The modest gap suggests that nonlinear interactions exist but are not dramatically stronger than linear effects for this feature set. |

### Phase 6 Complete

**Test-set evaluation:**
- [x] Test set unlocked and evaluated (first and only use)
- [x] Test AUC: 0.6930 (vs CV AUC 0.7025 -- less than 0.01 drop)
- [x] Test Recall: 0.6211 at default 0.5 threshold
- [x] Confusion matrix: 613 TP, 4,577 FP, 374 FN, 8,524 TN
- [x] ROC curves for all 3 tuned models on test set (XGB 0.6930, RF 0.6908, LR 0.6905)
- [x] Precision-recall curve with average precision (AP = 0.1565)
- [x] Threshold optimization: optimal F1 at threshold 0.50 (F1 = 0.1985); recall >= 0.70 at threshold 0.46

**Interpretability:**
- [x] SHAP importance plot -- top 3: time_in_hospital, number_inpatient, discharge_disposition_id_1
- [x] SHAP dot plot -- directional effects for top 20 features
- [x] SHAP waterfall plots for 3 individual patients (high-risk 0.896, low-risk 0.098, borderline 0.500)
- [x] Clinical risk factor summary -- 10 features in plain English with actionable recommendations

**Business impact:**
- [x] Cost-benefit simulation at 7 thresholds for 10,000-discharge hospital
- [x] Net benefit positive at all thresholds; optimal at threshold 0.30 (~$4.6M/year)
- [x] Operating threshold recommendation: 0.30 (92% recall, $4.6M net benefit)

**Hypotheses:**
- [x] 4 EDA hypotheses revisited -- 1 partially confirmed, 1 not confirmed, 2 confirmed

**Next: Phase 7 -- Documentation and Presentation**

---

## Phase 7: Documentation and Final Review

### 7.1 Notebook Summary

This notebook implements an end-to-end machine learning pipeline for predicting 30-day hospital readmission among diabetic inpatients, following the CRISP-DM methodology.

**Pipeline overview:**
- **Data:** 101,766 encounters from 130 US hospitals (UCI Diabetes 130-US dataset, Strack et al. 2014), cleaned to ~70,000 unique patients
- **Target:** Binary classification -- readmitted within 30 days (yes/no), ~11% positive rate
- **Features:** 31 input features (11 numeric, 16 categorical, 4 binary engineered) expanding to 148 after one-hot encoding
- **Models evaluated:** Logistic Regression, Random Forest, XGBoost, SVM
- **Final model:** XGBoost (tuned) -- AUC 0.693 on held-out test set
- **Business case:** $4.6M annual net benefit at threshold 0.30 for a 10,000-discharge hospital

**Notebook structure:**
| Phase | Cells | Description |
|-------|-------|-------------|
| Phase 1 | 0-35 | Data profiling: schema, target distribution, missing data, duplicates |
| Phase 2 | 36-80 | EDA: correlations, categorical deep-dives, 5 statistical tests, interactions |
| Phase 3 | 81-128 | Cleaning (8 steps), feature engineering (7 features), sklearn Pipeline |
| Bridge | 129-134 | CRISP-DM mapping, stakeholders, success metrics, imbalance strategy, flowchart |
| Phase 4 | 135-155 | Baseline training: 4 models x 5-fold CV, imbalance comparison, fold stability |
| Phase 5 | 156-181 | Hyperparameter tuning: 2-stage (RandomizedSearch + GridSearch), model selection |
| Phase 6 | 182-210 | Test-set evaluation, SHAP analysis, threshold optimization, business impact |
| Phase 7 | 211+ | Documentation and final review |

### 7.2 Limitations and Future Work

**Limitations of this analysis:**

1. **Dataset age (1999-2008):** Clinical practices, medication protocols, and hospital workflows have evolved since this data was collected. A model trained on this data would need retraining on current EHR data before clinical deployment. The HRRP business case remains current (penalties assessed annually since 2013), but the specific model coefficients may not transfer.

2. **Missing social determinants:** Housing stability, transportation access, caregiver support, health literacy, and socioeconomic status are among the strongest predictors of hospital readmission but are absent from this dataset. This likely explains the modest AUC ceiling (~0.70) -- clinical data alone cannot fully predict a phenomenon driven substantially by post-discharge social circumstances.

3. **Single-encounter deduplication:** Keeping only one encounter per patient (the longest stay) discards potentially informative longitudinal patterns. An alternative approach using time-series features from sequential encounters could capture disease progression but would require more complex modeling (e.g., recurrent neural networks).

4. **ICD-9 to ICD-10 transition:** The dataset uses ICD-9 codes (pre-2015). Current US hospitals use ICD-10, which has different code structure and granularity. The 9-category clinical grouping would need remapping for deployment on modern data.

5. **Threshold sensitivity:** The recommended operating threshold (0.30) flags ~75% of all patients for intervention. In a resource-constrained hospital, a higher threshold may be operationally necessary even if it reduces recall. The business impact simulation assumes unlimited intervention capacity.

**Future work:**

- **Validate on current data:** Retrain the pipeline on post-2015 EHR data with ICD-10 codes and compare feature importance rankings to this analysis.
- **Incorporate social determinants:** If available, add SDOH features (zip-code-level deprivation indices, insurance type detail, language barriers) and measure the AUC improvement.
- **Explore calibration:** The current model produces well-ranked probabilities (good AUC) but may not be well-calibrated -- Platt scaling or isotonic regression could improve probability estimates for clinical communication.
- **Deploy as SMART on FHIR app:** Package the model as a FHIR-integrated discharge risk tool that surfaces patient-level SHAP explanations at the point of care.

### 7.3 References

1. Strack, B., DeShazo, J., Gennings, C., Olmo, J.L., Ventura, S., Cios, K.J., and Clore, J.N. (2014). "Impact of HbA1c Measurement on Hospital Readmission Rates: Analysis of 70,000 Clinical Database Patient Records." *BioMed Research International*, 2014, Article ID 781670. doi:10.1155/2014/781670

2. Clore, J., Cios, K., DeShazo, J., and Strack, B. (2014). Diabetes 130-US Hospitals for Years 1999-2008 [Dataset]. UCI Machine Learning Repository. doi:10.24432/C5230J

3. CMS Hospital Readmissions Reduction Program. Centers for Medicare and Medicaid Services. https://www.cms.gov/medicare/quality/value-based-programs/hospital-readmissions

4. Masnoon, N., Shakib, S., Kalisch-Ellett, L., and Caughey, G.E. (2017). "What is polypharmacy? A systematic review of definitions." *BMC Geriatrics*, 17(1), 230. doi:10.1186/s12877-017-0621-2

5. Lundberg, S.M. and Lee, S.I. (2017). "A Unified Approach to Interpreting Model Predictions." *Advances in Neural Information Processing Systems*, 30.

In [ ]:
# Final notebook statistics
import json
import os

# Count cells
with open('DS_Capstone_Readmission.ipynb') as f:
    notebook = json.load(f)

total_cells = len(notebook['cells'])
md_cells = sum(1 for c in notebook['cells'] if c['cell_type'] == 'markdown')
code_cells = sum(1 for c in notebook['cells'] if c['cell_type'] == 'code')

# Count figures
fig_count = len([f for f in os.listdir('./figures') if f.endswith('.png')])

# Count saved models
model_count = len([f for f in os.listdir('./models') if f.endswith('.joblib')])

print("=" * 50)
print("FINAL NOTEBOOK STATISTICS")
print("=" * 50)
print(f"  Total cells:     {total_cells}")
print(f"  Markdown cells:  {md_cells}")
print(f"  Code cells:      {code_cells}")
print(f"  Figures saved:   {fig_count}")
print(f"  Models saved:    {model_count}")
print(f"  Phases:          7 (Profiling, EDA, Pipeline, Baselines, Tuning, Evaluation, Documentation)")
print(f"")
print(f"  Final model:     XGBoost (tuned)")
print(f"  Test AUC:        0.6930")
print(f"  Test Recall:     0.6211")
print(f"  Business Impact: ~$4.6M net benefit/year at threshold 0.30")
print(f"")
print(f"  NOTEBOOK COMPLETE")
print("=" * 50)

### Notebook Complete

All seven CRISP-DM phases are documented in this notebook. The pipeline is reproducible -- a Restart and Run All will regenerate all outputs from the raw CSV. The final XGBoost model is saved at `./models/final_model.joblib` and can be loaded for inference on new data with `joblib.load()`.

### Save Phase 6 Artifacts

In [ ]:
# Save Phase 6 final results
phase6_results = {
    'test_auc': test_metrics['auc'],
    'test_recall': test_metrics['recall'],
    'test_precision': test_metrics['precision'],
    'test_f1': test_metrics['f1'],
    'best_threshold': best_f1_threshold,
}

with open(f'{ARTIFACTS_DIR}/phase6_final_results.pkl', 'wb') as f:
    pickle.dump(phase6_results, f)

print('Phase 6 artifacts saved')
print(f'  phase6_final_results.pkl')
print(f'  Test AUC:    {test_metrics["auc"]:.4f}')
print(f'  Test Recall: {test_metrics["recall"]:.4f}')